In [1]:
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from pathlib import Path
from tqdm import tqdm
import os
import re

# --- 1. 설정 (Configuration) ---
print("1. 설정을 초기화합니다...")

# [사용자 설정] 파일 경로들을 지정합니다.
DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
BASE_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
# 최종적으로 저장될 제출 파일 이름
FINAL_SUBMISSION_PATH = "paligemma_submission.csv"

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. 설정을 초기화합니다...


In [2]:
# PaliGemma 모델 ID
MODEL_ID = "google/paligemma-3b-mix-224"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")

# --- 2. PaliGemma 모델 및 프로세서 로드 ---
print(f"\n2. PaliGemma 모델({MODEL_ID})을 로드합니다...")
# bfloat16을 사용하여 메모리 사용량을 줄입니다.
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto", # 자동으로 GPU 할당
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("모델 로드 완료.")

사용 디바이스: cuda

2. PaliGemma 모델(google/paligemma-3b-mix-224)을 로드합니다...


Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.30s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


모델 로드 완료.


In [3]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 2,923,466,480개 (2923.47M)
학습 가능한 파라미터: 2,923,466,480개 (2923.47M)


In [5]:


# --- 3. 데이터 로드 ---
print("\n3. 데이터 로드를 시작합니다...")
try:
    df_test = pd.read_csv(DEV_TEST_CSV_PATH)
    df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
except FileNotFoundError as e:
    print(f"오류: 필요한 파일을 찾을 수 없습니다: {e}")
    exit()

# --- 4. 추론 루프 (PaliGemma 방식) ---
print(f"\n4. 총 {len(df_test)}개의 데이터에 대한 추론을 시작합니다...")
# --- 루프 시작 전, 인덱스를 레이블로 변환할 딕셔너리 정의 ---
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

# tqdm을 사용하여 진행 상황을 시각적으로 표시
for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0], desc="추론 진행률"):
    
    test_id = row['ID']
    relative_img_path = row['img_path']
    image_filename = os.path.basename(relative_img_path)
    full_image_path = Path(BASE_IMAGE_DIR) / image_filename

    # --- 개선점: 질문(Question)과 선택지(Choice)를 결합 ---
    question = row['Question']
    # 각 선택지를 "질문: 선택지" 형태의 완전한 문장으로 만듭니다.
    choices = [
        f"Question: {question} Answer: {row['A']}",
        f"Question: {question} Answer: {row['B']}",
        f"Question: {question} Answer: {row['C']}",
        f"Question: {question} Answer: {row['D']}"
    ]

    try:
        image = Image.open(full_image_path).convert("RGB")
        
        # processor에 이미지와 재구성된 선택지 문장들을 함께 전달
        inputs = processor(images=image, text=choices, return_tensors="pt", padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
        
        # 이미지와 각 선택지 문장 간의 유사도 점수
        logits_per_image = outputs.logits_per_image.cpu()
        
        # 가장 높은 점수를 가진 선택지의 '인덱스' 찾기 (0, 1, 2, 3 중 하나)
        predicted_index = logits_per_image.argmax(-1).item()
        
        # 인덱스를 'A', 'B', 'C', 'D' 문자로 변환
        prediction_label = index_to_label[predicted_index]

        # df_submission에서 현재 ID와 일치하는 행을 찾아 'answer' 컬럼을 업데이트
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = prediction_label

    except FileNotFoundError:
        print(f"\n경고: ID '{test_id}'의 이미지 파일을 찾을 수 없습니다. 경로: {full_image_path}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'
    except Exception as e:
        print(f"\n오류: ID '{test_id}' 처리 중 예외 발생: {e}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'


# --- 5. 결과 저장 및 출력 ---
df_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)
print("\n" + "="*50)
print("추론이 완료되었습니다.")
print(f"결과가 {FINAL_SUBMISSION_PATH} 파일에 저장되었습니다.")
print("\n최종 제출 파일 미리보기 (상위 5개):")
print(df_submission.head())
print("="*50)


3. 데이터 로드를 시작합니다...

4. 총 60개의 데이터에 대한 추론을 시작합니다...


추론 진행률:   2%|▏         | 1/60 [00:00<00:11,  5.28it/s]


오류: ID 'TEST_000' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
추론 진행률:   5%|▌         | 3/60 [00:00<00:05, 11.06it/s]You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer h


오류: ID 'TEST_001' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_002' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_003' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'


...


오류: ID 'TEST_048' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_049' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_050' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_051' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
추론 진행률:  90%|█████████ | 54/60 [00:03<00:00, 18.78it/s]You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer 


오류: ID 'TEST_052' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_053' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_054' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_055' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
추론 진행률:  97%|█████████▋| 58/60 [00:03<00:00, 19.06it/s]You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer 


오류: ID 'TEST_056' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_057' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_058' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

오류: ID 'TEST_059' 처리 중 예외 발생: 'PaliGemmaCausalLMOutputWithPast' object has no attribute 'logits_per_image'

추론이 완료되었습니다.
결과가 paligemma_submission.csv 파일에 저장되었습니다.

최종 제출 파일 미리보기 (상위 5개):
         ID answer
0  TEST_000  Error
1  TEST_001  Error
2  TEST_002  Error
3  TEST_003  Error
4  TEST_004  Error
